# 🧪 Sesión 22 · Modelado Relacional (1:1, 1:N, N:M) con ORMs

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Cómo modelas relaciones sin escribir JOIN a mano?

---

## 🖥️ Así se verá el resultado final

```http
POST /api/autores  {"nombre":"Gabriel García Márquez"}
HTTP/1.1 201 Created
{"id":1,"nombre":"Gabriel García Márquez"}

POST /api/autores/1/libros  {"titulo":"Cien años de soledad","anio":1967}
HTTP/1.1 201 Created
{"id":1,"titulo":"Cien años de soledad","anio":1967,"autor_id":1}

GET /api/autores/1
HTTP/1.1 200 OK
{"id":1,"nombre":"Gabriel García Márquez",
 "libros":[{"id":1,"titulo":"Cien años de soledad"}]}
```

_El ORM mantiene la relación: tú trabajas con objetos, él escribe el JOIN._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Definir modelos ORM con relaciones 1:N.
- Crear un libro asociado a un autor.
- Verificar la integridad referencial en la respuesta.

## ✅ Criterios de aceptación

El verificador ejecutará los casos HTTP y comprobará los archivos del contrato. Todo debe quedar en verde:

- [ ] Se crea el autor → HTTP 201
- [ ] Se crea el libro asociado → HTTP 201
- [ ] El autor devuelve sus libros → HTTP 200
- [ ] El listado de libros funciona → HTTP 200
- [ ] La relación ORM está declarada

## 🧩 El reto

Modela autores y libros con SQLAlchemy, una relación 1:N y endpoints que respeten la integridad referencial.

**Requisitos del contrato**

- El modelo Autor tiene id y nombre.
- El modelo Libro tiene id, titulo, anio y una llave foránea autor_id.
- La relación Autor.libros está declarada con relationship.
- POST /api/autores y POST /api/autores/{id}/libros responden 201.
- GET /api/autores/{id} devuelve el autor con sus libros anidados.

> **Pista conceptual:** En SQLAlchemy, `ForeignKey('autores.id')` y `relationship('Libro', back_populates='autor')` describen la relación. La base se crea con `Base.metadata.create_all(engine)`.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion22.md` con el diagrama entidad-relación.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [1]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [2]:
%%writefile app.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from sqlalchemy import create_engine, Column, Integer, String, ForeignKey
from sqlalchemy.orm import declarative_base, relationship, sessionmaker

DATABASE_URL = "sqlite:///./biblioteca.db"
engine = create_engine(DATABASE_URL, connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(bind=engine, autocommit=False, autoflush=False)
Base = declarative_base()

class Autor(Base):
    __tablename__ = "autores"
    id = Column(Integer, primary_key=True, index=True)
    nombre = Column(String, nullable=False)
    libros = relationship("Libro", back_populates="autor", cascade="all, delete-orphan")

class Libro(Base):
    __tablename__ = "libros"
    id = Column(Integer, primary_key=True, index=True)
    titulo = Column(String, nullable=False)
    anio = Column(Integer, nullable=False)
    autor_id = Column(Integer, ForeignKey("autores.id"), nullable=False)
    autor = relationship("Autor", back_populates="libros")

Base.metadata.create_all(bind=engine)

app = FastAPI(title="Relación autor -> libros")

class AutorIn(BaseModel):
    nombre: str

class LibroIn(BaseModel):
    titulo: str
    anio: int

def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

@app.post("/api/autores", status_code=201)
def crear_autor(datos: AutorIn):
    Base.metadata.create_all(bind=engine)
    db = SessionLocal()
    try:
        autor = Autor(nombre=datos.nombre)
        db.add(autor)
        db.commit()
        db.refresh(autor)
        return {"id": autor.id, "nombre": autor.nombre}
    finally:
        db.close()

@app.post("/api/autores/{autor_id}/libros", status_code=201)
def crear_libro(autor_id: int, datos: LibroIn):
    Base.metadata.create_all(bind=engine)
    db = SessionLocal()
    try:
        autor = db.query(Autor).filter(Autor.id == autor_id).first()
        if not autor:
            raise HTTPException(status_code=404, detail="Autor no encontrado")
        libro = Libro(titulo=datos.titulo, anio=datos.anio, autor_id=autor_id)
        db.add(libro)
        db.commit()
        db.refresh(libro)
        return {"id": libro.id, "titulo": libro.titulo, "anio": libro.anio, "autor_id": libro.autor_id}
    finally:
        db.close()

@app.get("/api/autores/{autor_id}")
def obtener_autor(autor_id: int):
    Base.metadata.create_all(bind=engine)
    db = SessionLocal()
    try:
        autor = db.query(Autor).filter(Autor.id == autor_id).first()
        if not autor:
            raise HTTPException(status_code=404, detail="Autor no encontrado")
        return {"id": autor.id, "nombre": autor.nombre, "libros": [{"id": l.id, "titulo": l.titulo} for l in autor.libros]}
    finally:
        db.close()

@app.get("/api/libros")
def listar_libros():
    Base.metadata.create_all(bind=engine)
    db = SessionLocal()
    try:
        libros = db.query(Libro).all()
        return {"libros": [{"id": l.id, "titulo": l.titulo, "anio": l.anio, "autor_id": l.autor_id} for l in libros]}
    finally:
        db.close()

Writing app.py


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Framework, versión y restricciones del entorno. |
| **Rol** | Qué papel técnico debe asumir el modelo. |
| **Especificación** | Rutas, entradas, salidas, validaciones y códigos HTTP. |
| **Output** | Qué formato exacto esperas de vuelta. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [3]:
from verificar import probar_http

casos = [{'descripcion': 'Se crea el autor',
  'metodo': 'POST',
  'ruta': '/api/autores',
  'json': {'nombre': 'Gabriel García Márquez'},
  'estado': 201,
  'json_contiene': {'id': 1, 'nombre': 'Gabriel García Márquez'}},
 {'descripcion': 'Se crea el libro asociado',
  'metodo': 'POST',
  'ruta': '/api/autores/1/libros',
  'json': {'titulo': 'Cien años de soledad', 'anio': 1967},
  'estado': 201,
  'json_contiene': {'id': 1, 'titulo': 'Cien años de soledad', 'autor_id': 1}},
 {'descripcion': 'El autor devuelve sus libros',
  'metodo': 'GET',
  'ruta': '/api/autores/1',
  'estado': 200,
  'json_contiene': {'id': 1,
                    'nombre': 'Gabriel García Márquez',
                    'libros': [{'id': 1, 'titulo': 'Cien años de soledad'}]}},
 {'descripcion': 'El listado de libros funciona',
  'metodo': 'GET',
  'ruta': '/api/libros',
  'estado': 200,
  'json_contiene': {'libros': [{'id': 1, 'titulo': 'Cien años de soledad'}]}}]

probar_http(casos, archivo="app.py", limpiar=['biblioteca.db'], env={})


VERIFICACIÓN DEL RESULTADO · 4 casos · app.py
✅ 01. Se crea el autor
✅ 02. Se crea el libro asociado
✅ 03. El autor devuelve sus libros
✅ 04. El listado de libros funciona
------------------------------------------------------------------------------
RESULTADO: 4/4 casos correctos. ¡Resultado logrado!


In [4]:
from verificar import revisar_archivo, resumen

resultados = []
resultados.append(revisar_archivo("app.py", regex=["relationship\\(|ForeignKey\\("], descripcion="La relación ORM está declarada"))
resumen(resultados)


✅ La relación ORM está declarada
------------------------------------------------------------------------------
RESULTADO: 1/1 comprobaciones correctas. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion22.md` a `auditorias/auditoria_sesion22.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   - ¿Qué error detectó la verificación que tú no habías previsto?
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
